In [ ]:
!pip install scipy

In [ ]:
import os
from pprint import pprint

import pandas as pd
import numpy as np
import pyreadstat
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from scipy import stats

In [ ]:
# %% [Load data]
df, metadata = pyreadstat.read_sav('../../data/0_raw/ZM_LFS_DATASET2024_Annual_10percent.sav')

# %% [Select columns of interest]
# This cell was written after investigating the above result and picking
# column names that were likely to contain interesting data

cols = [
    # --- Demographics & Background ---
    "Is ... Male or Female?",
    "How old was ... at (his/her) last birthday?",
    "What is the highest grade/level of education that ... has successfully completed?",
    "What is ...'s current marital status?",
    "What is ...'s relationship to the head of the household?",
    "1. Province",
    "2. District",

    # --- Employment & Work ---
    "In the main job/business that (NAME) has, is she/he...",
    "INDUSTRY",
    "Occupation",
    "How many hours does (NAME) usually work per week in his/her...? Main job",
    "How many hours does (NAME) usually work per week in his/her...? OVERALL TOTAL",
    "What is the frequency of .....'s income/earnings in his/her main job?",
    "Would (NAME) want to work more hours per week than usually worked, provided the extra hours are paid?",
    "Is ?. employed on the basis of a written contract or an oral agreement?",

    # --- Income & Earnings ---
    "What is your annually/monthly/weekly/daily/hourly wage or salary before deductions?",
    "What are your annual/monthly/weekly/daily/hourly earnings after expenses?",
    "At what age did NAME start work for the first time in his /her life",

    # --- Time Use: Household Activities ---
    "During the last 7 days how much time did  (NAME) spend on Cleaning the house, washing clothes, cooking or shopping for the household",
    "During the last 7 days how much time did  (NAME) spend on Fetching water from natural or public sources for use by the household",
    "During the last 7 days how much time did (NAME) spend on Collecting firewood or other natural products for use as fuel by the household",
    "In the last 7 days, how much time did (NAME) spend on Leisure e.g., playing sports, watching TV etc.?",
    "In the last 7 days, how much time did (NAME) spend on Personal care e.g bathing, eating and sleeping?",
    "In the last 7 days, how much time did name spend travelling from home to\xa0place\xa0of\xa0work",

    # --- Time Use: Hours by Day of Week (Main Job) ---
    "Thinking about the last 7 Days, how many hours  ?? work his/her on Monday Main job?",
    "Thinking about the last 7 Days, how many hours  ?? work his/her on Tuesday Main job?",
    "Thinking about the last 7 Days, how many hours  ?? work his/her on Wednesday Main job?",
    "Thinking about the last 7 Days, how many hours  ?? work his/her on Thursday Main job?",
    "Thinking about the last 7 Days, how many hours  ?? work his/her on Friday Main job?",
    "Thinking about the last 7 Days, how many hours  ?? work his/her on Saturday Main job?",
    "Thinking about the last 7 Days, how many hours  ?? work his/her on Sunday Main job?",

    # --- Financial Inclusion ---
    "P.20. Do you own a mobile phone",
    "P.21. Do you have a mobile money account in your own name",
    "P.23. How often do you use mobile money?",
    "P.25.  On a scale of 1 to 4, Do you find mobile money services to be cheap or expensive?",
    "P.30.A Savings at a bank",
    "P.30.G.Savings with savings group",
    "P.30.D.Savings that you keep on your mobile phone",
    "What method do you mainly use to pay for food/groceries?",

    # --- Education ---
    "Can... read and write in any language?",
    "Has... ever attended school?",
    "Is (NAME) currently attending school?",
    "Have (NAME) ever repeated any level of schooling any point in time?",
    "At what age did (NAME) begin school?",
]

# For each label we pick the original column name as it appears in df.
names_to_labels = metadata.column_names_to_labels
names_to_labels_reduced = {}
names = []
for col in cols:
    for name, label in names_to_labels.items():
        if label != col:
            continue
        names.append(name)
        names_to_labels_reduced[name] = label
pprint(names_to_labels_reduced)

# %% [Filter dataframe to columns of interest]
df = df[names]

# %% [Extract value labels from metadata]
variable_value_labels = metadata.variable_value_labels

In [ ]:
normal = stats.norm(loc=0, scale=1)

In [ ]:
arr = normal.rvs(size=100000, random_state=42)
pd.DataFrame(arr).hist(bins=50)

In [ ]:
normal.cdf(0)

In [ ]:
df_interval = df[['A2', 'A3', 'PROV']].dropna()
confindence_level = 0.95

for prov in df_interval['PROV'].unique():
    df_province = df_interval[df_interval['PROV'] == prov]

    sample_mean = np.mean(df_province['A3'])
    standard_error = stats.sem(df_province['A3'])
    degrees_of_freedeom = len(df_province['A3']) - 1
    i_low, i_high = stats.t.interval(
        confindence_level,
        df=degrees_of_freedeom,
        loc=sample_mean,
        scale=standard_error
    )
    print(f"""Province {prov} has mean age {sample_mean:.2f}, 
    CI95 ({i_low:.2f}, {i_high:.2f}),
    and sample size {degrees_of_freedeom}""")
        

In [ ]:
baseline = 100
year_1_growth = 1.5
year_2_grwoth = 0.5
baseline * year_1_growth * year_2_grwoth

In [ ]:
classical_mean = (year_1_growth + year_2_grwoth)/2
classical_mean

In [ ]:
gmean = stats.gmean([year_1_growth, year_2_grwoth])
baseline * gmean * gmean

In [ ]:
def descriptive_report(data, name="Variable"):
    """Print a comprehensive descriptive statistics report."""
    desc = stats.describe(data)
    
    print(f"=== Descriptive Report: {name} ===")
    print(f"N:               {desc.nobs}")
    print(f"Mean:            {desc.mean:,.2f}")
    print(f"5% Trimmed Mean: {stats.trim_mean(data, 0.05):,.2f}")
    print(f"Median:          {np.median(data):,.2f}")
    print(f"Std Dev:         {np.sqrt(desc.variance):,.2f}")
    print(f"Min:             {desc.minmax[0]:,.2f}")
    print(f"Max:             {desc.minmax[1]:,.2f}")
    print(f"Skewness:        {desc.skewness:.3f}")
    print(f"Excess Kurtosis: {desc.kurtosis:.3f}")
    print(f"Std Error:       {stats.sem(data):,.2f}")
